In [ ]:
# ==========================================================
# CLOUD-BASED STUDENT RESULT MANAGEMENT SYSTEM
# GOOGLE COLAB - ONE CELL
# ==========================================================

!pip -q install flask

from flask import Flask, request, redirect, render_template_string
import threading
import time
from google.colab.output import eval_js
from IPython.display import display, HTML as IPHTML

app = Flask(__name__)

students = []

# ==========================================================
# HTML
# ==========================================================

PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>Student Result Management</title>

<style>

body {
    margin:0;
    font-family:Arial;
    background:#eef5ff;
}

header {
    background:#2455a4;
    color:white;
    text-align:center;
    padding:25px;
}

nav {
    background:#173d78;
    padding:15px;
    text-align:center;
}

nav a {
    color:white;
    text-decoration:none;
    margin:20px;
    font-weight:bold;
}

.container {
    width:90%;
    max-width:1000px;
    margin:30px auto;
}

.card {
    background:white;
    padding:25px;
    border-radius:12px;
    box-shadow:0 3px 10px #bbb;
    margin-bottom:25px;
}

input {
    width:100%;
    padding:12px;
    margin:8px 0 15px;
    box-sizing:border-box;
}

button {
    background:#2455a4;
    color:white;
    border:none;
    padding:12px 20px;
    border-radius:6px;
    cursor:pointer;
}

table {
    width:100%;
    border-collapse:collapse;
}

th,td {
    border:1px solid #ccc;
    padding:10px;
    text-align:center;
}

th {
    background:#2455a4;
    color:white;
}

.success {
    background:#dff0d8;
    color:green;
    padding:15px;
    margin-bottom:20px;
}

.pass {
    color:green;
    font-weight:bold;
}

.fail {
    color:red;
    font-weight:bold;
}

.delete {
    background:#d9534f;
}

</style>
</head>

<body>

<header>
<h1>🎓 Cloud Student Result System</h1>
<p>Cloud-Based Student Result Management System</p>
</header>

<nav>
<a href="/">Home</a>
<a href="/add">Add Result</a>
<a href="/results">View Results</a>
<a href="/search">Search Result</a>
</nav>

<div class="container">

{% if message %}
<div class="success">{{ message }}</div>
{% endif %}


{% if page == "home" %}

<div class="card">

<h2>Welcome</h2>

<p>
This system is used to manage student examination
results using a cloud-based web application.
</p>

<h3>Features</h3>

<ul>
<li>Add Student Result</li>
<li>Calculate Total Marks</li>
<li>Calculate Percentage</li>
<li>Generate Grade</li>
<li>Pass / Fail Status</li>
<li>Search Student Result</li>
<li>Delete Result</li>
</ul>

<a href="/add">
<button>Add Student Result</button>
</a>

</div>


{% elif page == "add" %}

<div class="card">

<h2>➕ Add Student Result</h2>

<form method="POST" action="/add">

<label>Register Number</label>
<input type="text" name="regno" required>

<label>Student Name</label>
<input type="text" name="name" required>

<label>Department</label>
<input type="text" name="department" required>

<label>Cloud Computing</label>
<input type="number" name="s1" min="0" max="100" required>

<label>Computer Networks</label>
<input type="number" name="s2" min="0" max="100" required>

<label>Web Technology</label>
<input type="number" name="s3" min="0" max="100" required>

<label>Database Management</label>
<input type="number" name="s4" min="0" max="100" required>

<label>Artificial Intelligence</label>
<input type="number" name="s5" min="0" max="100" required>

<button type="submit">
Save Result
</button>

</form>

</div>


{% elif page == "results" %}

<div class="card">

<h2>📋 Student Results</h2>

{% if students %}

<table>

<tr>
<th>Reg No</th>
<th>Name</th>
<th>Department</th>
<th>Total</th>
<th>Percentage</th>
<th>Grade</th>
<th>Status</th>
<th>Action</th>
</tr>

{% for s in students %}

<tr>

<td>{{s.regno}}</td>
<td>{{s.name}}</td>
<td>{{s.department}}</td>
<td>{{s.total}}/500</td>
<td>{{s.percentage}}%</td>
<td>{{s.grade}}</td>

<td>
{% if s.status == "PASS" %}
<span class="pass">PASS</span>
{% else %}
<span class="fail">FAIL</span>
{% endif %}
</td>

<td>
<a href="/delete/{{s.regno}}">
<button class="delete">Delete</button>
</a>
</td>

</tr>

{% endfor %}

</table>

{% else %}

<p>No results available.</p>

{% endif %}

</div>


{% elif page == "search" %}

<div class="card">

<h2>🔎 Search Student Result</h2>

<form method="POST" action="/search">

<input
type="text"
name="regno"
placeholder="Enter Register Number"
required
>

<button type="submit">
Search
</button>

</form>

{% if student %}

<hr>

<h3>🎓 Student Result</h3>

<p><b>Register Number:</b> {{student.regno}}</p>
<p><b>Name:</b> {{student.name}}</p>
<p><b>Department:</b> {{student.department}}</p>

<hr>

<p>Cloud Computing: {{student.s1}}</p>
<p>Computer Networks: {{student.s2}}</p>
<p>Web Technology: {{student.s3}}</p>
<p>Database Management: {{student.s4}}</p>
<p>Artificial Intelligence: {{student.s5}}</p>

<hr>

<h3>Total: {{student.total}} / 500</h3>
<h3>Percentage: {{student.percentage}}%</h3>
<h3>Grade: {{student.grade}}</h3>

<h3>
Status:
{% if student.status == "PASS" %}
<span class="pass">PASS</span>
{% else %}
<span class="fail">FAIL</span>
{% endif %}
</h3>

{% elif searched %}

<p class="fail">❌ Student not found.</p>

{% endif %}

</div>

{% endif %}

</div>

</body>
</html>
"""


# ==========================================================
# GRADE
# ==========================================================

def grade(marks):

    if min(marks) < 40:
        return "F"

    percentage = sum(marks) / 5

    if percentage >= 90:
        return "A+"
    elif percentage >= 80:
        return "A"
    elif percentage >= 70:
        return "B"
    elif percentage >= 60:
        return "C"
    elif percentage >= 50:
        return "D"
    else:
        return "E"


# ==========================================================
# HOME
# ==========================================================

@app.route("/")
def home():

    return render_template_string(
        PAGE,
        page="home",
        students=students,
        student=None,
        searched=False,
        message=None
    )


# ==========================================================
# ADD
# ==========================================================

@app.route("/add", methods=["GET","POST"])
def add():

    if request.method == "POST":

        regno = request.form["regno"]

        # Check duplicate

        for s in students:

            if s["regno"] == regno:

                return render_template_string(
                    PAGE,
                    page="add",
                    students=students,
                    student=None,
                    searched=False,
                    message="Register number already exists!"
                )

        marks = [
            int(request.form["s1"]),
            int(request.form["s2"]),
            int(request.form["s3"]),
            int(request.form["s4"]),
            int(request.form["s5"])
        ]

        total = sum(marks)

        percentage = round(total / 5, 2)

        g = grade(marks)

        status = "FAIL" if g == "F" else "PASS"

        students.append({

            "regno":regno,

            "name":request.form["name"],

            "department":request.form["department"],

            "s1":marks[0],
            "s2":marks[1],
            "s3":marks[2],
            "s4":marks[3],
            "s5":marks[4],

            "total":total,

            "percentage":percentage,

            "grade":g,

            "status":status

        })

        return render_template_string(
            PAGE,
            page="home",
            students=students,
            student=None,
            searched=False,
            message="✅ Result saved successfully!"
        )

    return render_template_string(
        PAGE,
        page="add",
        students=students,
        student=None,
        searched=False,
        message=None
    )


# ==========================================================
# RESULTS
# ==========================================================

@app.route("/results")
def results():

    return render_template_string(
        PAGE,
        page="results",
        students=students,
        student=None,
        searched=False,
        message=None
    )


# ==========================================================
# SEARCH
# ==========================================================

@app.route("/search", methods=["GET","POST"])
def search():

    student = None
    searched = False

    if request.method == "POST":

        searched = True

        regno = request.form["regno"]

        for s in students:

            if s["regno"] == regno:

                student = s
                break

    return render_template_string(
        PAGE,
        page="search",
        students=students,
        student=student,
        searched=searched,
        message=None
    )


# ==========================================================
# DELETE
# ==========================================================

@app.route("/delete/<regno>")
def delete(regno):

    global students

    students = [
        s for s in students
        if s["regno"] != regno
    ]

    return redirect("/results")


# ==========================================================
# START SERVER
# ==========================================================

def run():

    app.run(
        host="0.0.0.0",
        port=7860,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(
    target=run,
    daemon=True
)

thread.start()

time.sleep(3)


# ==========================================================
# COLAB LINK
# ==========================================================

url = eval_js(
    "google.colab.kernel.proxyPort(7860)"
)

print("======================================")
print(" STUDENT RESULT MANAGEMENT SYSTEM")
print("======================================")
print("Server is running!")
print()
print(url)
print("======================================")


display(
    IPHTML(
        f'''
        <br>
        <a href="{url}" target="_blank">
        <button style="
        background:#2455a4;
        color:white;
        padding:15px 30px;
        border:0;
        border-radius:8px;
        font-size:18px;
        cursor:pointer;">
        🎓 OPEN STUDENT RESULT SYSTEM
        </button>
        </a>
        '''
    )
)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:7860
 * Running on http://172.28.0.12:7860
INFO:werkzeug:Press CTRL+C to quit


 STUDENT RESULT MANAGEMENT SYSTEM
Server is running!

https://7860-m-s-kkb-ase1a1-6q4adz1flnyt-a.asia-east1-1.prod.colab.dev
